# Question 4
Add a second target (staging or prod) to your databricks.yml with a different workspace host and run_as service principal, and deploy to it.




1. Created the Databricks Bundle in the Development Workspace.

2. Got access to the Production Workspace using the browser.

3. Gave the Deployer CLI access to the Production Workspace.  

```databricks auth login --host  prods-url --profile prod-profile-name```


4. Created a Service Principal in the Production Workspace.

5. Gave the required permissions to the Service Principal:
   - Bundle permissions
   - Catalog permissions
   - Required resource permissions

6. Gave the Deployer user permission to use the Service Principal.

7. Added the Service Principal in `run_as` inside `databricks.yml`.

```
# This is a Declarative Automation Bundle definition for Day_9_assignment.
# See https://docs.databricks.com/dev-tools/bundles/index.html for documentation.
bundle:
  name: Day_9_assignment
  uuid: eea8c173-a2bc-4815-aeb5-ecb61425efa6

include:
  - resources/*.yml

# Variable declarations. These variables are assigned in the dev/prod targets below.
variables:
  catalog:
    description: The catalog to use
  schema:
    description: The schema to use

targets:
  dev:
    # The default target uses 'mode: development' to create a development copy.
    # - Deployed resources get prefixed with '[dev my_user_name]'
    # - Any job schedules and triggers are paused by default.
    # See also https://docs.databricks.com/dev-tools/bundles/deployment-modes.html.
    mode: development
    default: true
    workspace:
      host: https://dbc-2bd5e4bd-405a.cloud.databricks.com
    variables:
      catalog: dev
      schema: bronze
  prod:
    mode: production
    workspace:
      host: https://dbc-23ec29fd-2d98.cloud.databricks.com
      # We explicitly deploy to /Workspace/Users/adikamodiya@gmail.com to make sure we only have a single copy.
      root_path: /Workspace/Users/agrawaldeepak386@gmail.com/.bundle/${bundle.name}/${bundle.target}
    
    run_as:
      service_principal_name: 15096f92-eadd-4378-8955-79abbebbad27



    variables:
      catalog: prod
      schema: bronze
    permissions:
      - user_name: adikamodiya@gmail.com
        level: CAN_MANAGE
```

8. commands  
```databricks bundle validate```

```databricks  bundle deploy -t prod```

9. output 

```PS D:\Databricks\Day_9_assignment> databricks bundle deploy -t prod
Uploading bundle files to /Workspace/Users/agrawaldeepak386@gmail.com/.bundle/Day_9_assignment/prod/files...
Deploying resources...
Updating deployment state...
Deployment complete!
PS D:\Databricks\Day_9_assignment> ```




# Question 5
Configure M2M (service principal) authentication for the CLI and use it instead of your personal U2M
login for a deploy command.


## Overview
Configured Machine-to-Machine (M2M) authentication using a Databricks Service Principal instead of User-to-Machine (U2M) personal login to perform automated bundle deployments.

---

## Steps Taken

### Step 1: Create Service Principal & OAuth Secret
1. In Databricks Workspace, went to **Settings > Identity and access > Service principals**.
2. Created a new Service Principal named `day9-m2m-sp`.
3. Under **OAuth credentials**, clicked **Generate secret** and copied the **Client ID** and **Client Secret**.

---

### Step 2: Configure Local CLI Profile (`.databrickscfg`)
Opened `%USERPROFILE%\.databrickscfg` and added the M2M authentication profile:

```ini
[sp-deploy-profile]
host = [https://dbc-23ec29fd-2d98.cloud.databricks.com](https://dbc-23ec29fd-2d98.cloud.databricks.com)
client_id = <YOUR_CLIENT_ID>
client_secret = <YOUR_CLIENT_SECRET>
```

Step 3: Update databricks.yml Target Configuration
Updated the target workspace host and attached sp-deploy-profile in databricks.yml:

```targets:
  dev:
    mode: development
    default: true
    workspace:
      host: [https://dbc-23ec29fd-2d98.cloud.databricks.com](https://dbc-23ec29fd-2d98.cloud.databricks.com)
      profile: sp-deploy-profile

    variables:
      catalog: dev
      schema: bronze
      ```


Step 4: Verify M2M Connection
Ran the verification command to confirm CLI is authenticating as the Service Principal:

      ```databricks current-user me --profile sp-deploy-profile```

      

Step 5 : deploy command      

      ```databricks bundle deploy -t dev```

# Question 6
Write a GitHub Actions workflow that runs databricks bundle validate on every pull request, without
deploying anything.


# Databricks Asset Bundle Validation Workflow Setup

## Step 1: Isolated Branch Setup

- **Goal:** Keep the `main` branch non-initializable (untouched) while setting up the assignment target branch.

- **Actions Taken:**
  - Fetched the latest repository code locally.
  - Created and checked out a new target branch named `main-2` from `main`.
  - Created a dedicated feature branch named `validate-bundle-branch` off `main-2` to develop and test the setup.

---

## Step 2: Project & Asset Bundle Initialization

- **Goal:** Set up a Databricks Asset Bundle (DAB) inside a specific project folder.

- **Actions Taken:**
  - Created the isolated folder structure: `day_9_Q_6_bundle/my_project`.
  - Ran `databricks bundle init` inside `day_9_Q_6_bundle/my_project` using the `default-python` template.
  - Generated the `databricks.yml` configuration file pre-filled with the workspace host URL.

---

## Step 3: GitHub Actions Workflow Creation

- **Goal:** Automate bundle validation on Pull Requests targeting `main-2`.

- **Actions Taken:**
  - Created the GitHub Actions workflow file at `.github/workflows/bundle-validate.yml` in the repository root.
  - Configured the workflow trigger specifically for Pull Requests targeting `main-2`.

```yaml
on:
  pull_request:
    branches:
      - main-2

    


```
## Step 4: Workflow Tag & Action Name Troubleshooting

- **Goal:** Resolve GitHub runner setup failures for the Databricks CLI installer action.

- **Actions Taken:**
  - Fixed initial marketplace resolution errors by correcting the action reference to `databricks/setup-cli@main`.
  - Committed and pushed fixes across commits `5236b36`, `2aabe0f`, `de348ac`, `16209f9`, and `e29256d` on `validate-bundle-branch`.

---

## Step 5: Authentication & Secrets Configuration

- **Goal:** Provide workspace credentials securely to the GitHub Actions runner.

- **Actions Taken:**
  - Identified the authentication failure during `databricks bundle validate` execution (`default auth: cannot configure default credentials`).
  - Generated a Personal Access Token (PAT) inside the Databricks Workspace settings.
  - Added `DATABRICKS_TOKEN` under **Settings > Secrets and variables > Actions** in the GitHub repository settings.
  - Verified that `DATABRICKS_HOST` was pulled from the bundle configuration file.

---

## Step 6: Pull Request Validation & Merge

- **Goal:** Test CI execution, verify results, and integrate changes.

- **Actions Taken:**
  - Opened Pull Request #3 from `validate-bundle-branch` into `main-2`.
  - Re-ran the workflow on commit `e29256d`.
  - Verified that the check passed green (✓) with successful validation and zero errors.
  - Executed the **Merge pull request** action to sync `main-2` with `validate-bundle-branch` while leaving `main` completely untouched.

---

## `bundle-validate.yml`

```yaml
name: Databricks Bundle Validation 
 
on: 
  pull_request: 
    branches: 
      - main-2 
 
jobs: 
  validate: 
    runs-on: ubuntu-latest 
 
    steps: 
      - name: Checkout Repository 
        uses: actions/checkout@v4 
 
      - name: Setup Databricks CLI 
        uses: databricks/setup-cli@main 
 
      - name: Validate Databricks Bundle 
        working-directory: ./day_9_Q_6_bundle/my_project 
        env: 
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }} 
          DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN }} 
        run: | 
          databricks bundle validate





## Project Structure

```text
├── .github/
│   └── workflows/
│       └── bundle-validate.yml          # GitHub Actions CI workflow for main-2 PRs
│
├── day_9_Q_6_bundle/
│   └── my_project/                      # Databricks Asset Bundle directory
│       ├── .databricks/                 # Local Databricks CLI metadata (gitignored)
│       ├── resources/                   # Job and pipeline definitions
│       │   └── my_project_job.yml
│       ├── src/                         # Python source code / notebooks
│       │   └── main.py
│       ├── databricks.yml               # Bundle configuration file (host & targets)
│       └── README.md